# 🏥 Multilingual Medical Chatbot — Fine-Tuned v5
### Real datasets + IndicTrans2 translation for Indian languages

**Runtime → Change runtime type → T4 GPU**

## 📦 Dataset Strategy
There is **no dedicated Hindi/Telugu/Tamil/Kannada medical QA dataset** on HuggingFace.
The best approach (used by AI4Bharat, Sarvam, etc.) is:

1. **English medical datasets** (real, high-quality)
   - `ruslanmv/ai-medical-chatbot` — 250k doctor-patient dialogues
   - `lavita/medical-qa-datasets` (health_advice subset) — patient health Q&A
   - `medical_meadow_health_advice` — practical health advice

2. **Translate to Indian languages** using `ai4bharat/indictrans2` — the same translation
   pipeline used by AI4Bharat to build all Indic LLMs (Airavata, OpenHathi, etc.)

> ⚠️ Educational only. Always consult a real doctor.

In [ ]:
# ============================================================
# CELL 1: Install
# ── THE ONLY CORRECT FIX for bitsandbytes + triton.ops on Colab ──
#
# Root cause: bitsandbytes <= 0.44 has a hard dependency on
# triton.ops.matmul_perf_model which was REMOVED from triton 3.x.
# Fix: upgrade bitsandbytes to 0.45.2+ where the triton dependency
# was completely removed. Do NOT touch triton itself.
# ============================================================
import subprocess, sys, os

os.environ["BITSANDBYTES_NOWELCOME"] = "1"  # suppress spam

def pip(args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args)
    return r.returncode == 0

# Step 1: Upgrade bitsandbytes — this is the ONLY fix needed
print("[1/3] Installing bitsandbytes (latest)...")
pip(["bitsandbytes>=0.45.2"])

# Step 2: Core packages — pinned for stability
print("[2/3] Installing core packages...")
pip([
    "transformers==4.46.3",
    "accelerate==1.1.1",
    "peft==0.13.2",
    "trl==0.8.6",
    "sentencepiece",
    "protobuf",
    "datasets",
    "gradio",
])

# Step 3: Verify — check CUDA backend loaded correctly
print("[3/3] Verifying bitsandbytes CUDA backend...")
import importlib
import importlib.util

# Must restart kernel for new bitsandbytes to take effect if it was
# already imported earlier in the session
spec = importlib.util.find_spec("bitsandbytes")
if spec:
    import bitsandbytes as bnb
    v = bnb.__version__
    print(f"    bitsandbytes {v} found")
    from packaging.version import Version
    if Version(v) < Version("0.45.2"):
        print("    ⚠ Version too old — triton.ops error will occur!")
        print("    → RESTART RUNTIME and re-run this cell")
    else:
        print(f"    ✅ Version OK — triton.ops issue is fixed in {v}")
else:
    print("    ⚠ bitsandbytes not found after install")
    print("    → RESTART RUNTIME and re-run this cell")

print("
✅ Done! If you see any warning above → restart runtime first.")


In [ ]:
# ============================================================
# CELL 2: Load Translator
# Using facebook/mbart-large-50-many-to-many-mmt
#   ✅ Completely open — no gating, no approval needed
#   ✅ No special tokenizer package required
#   ✅ Native support: Hindi, Telugu, Tamil, Kannada + 46 more
#   ✅ Simple API: just set src_lang and forced_bos_token_id
# ============================================================
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import torch

TRANS_MODEL = "facebook/mbart-large-50-many-to-many-mmt"

# Language codes for mBART-50
LANG_CODES = {
    "hindi":   "hi_IN",
    "telugu":  "te_IN",
    "tamil":   "ta_IN",
    "kannada": "kn_IN",
}

print(f"Loading {TRANS_MODEL}...")
trans_tokenizer = MBart50TokenizerFast.from_pretrained(TRANS_MODEL)
trans_model = MBartForConditionalGeneration.from_pretrained(
    TRANS_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
).eval()
print("Translator loaded!")

def translate_batch(texts, tgt_lang_code, batch_size=8):
    """Translate a list of English texts to target Indian language."""
    all_translations = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        trans_tokenizer.src_lang = "en_XX"
        inputs = trans_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        ).to(trans_model.device)
        forced_bos = trans_tokenizer.lang_code_to_id[tgt_lang_code]
        with torch.no_grad():
            generated = trans_model.generate(
                **inputs,
                forced_bos_token_id=forced_bos,
                num_beams=4,
                max_length=256,
            )
        decoded = trans_tokenizer.batch_decode(generated, skip_special_tokens=True)
        all_translations.extend(decoded)
    return all_translations

# Quick test
print("
Translation test:")
tests = [
    ("I have fever and headache.", "hi_IN", "Hindi"),
    ("I have fever and headache.", "te_IN", "Telugu"),
    ("I have fever and headache.", "ta_IN", "Tamil"),
    ("I have fever and headache.", "kn_IN", "Kannada"),
]
for text, code, name in tests:
    result = translate_batch([text], code)
    print(f"  {name}: {result[0]}")
print("
✓ Translator ready!")


In [ ]:
# ============================================================
# CELL 3: Load English Medical Datasets
# ============================================================
from datasets import load_dataset
import random
random.seed(42)

english_qa = []

# ── Dataset 1: ruslanmv/ai-medical-chatbot (250k doctor-patient dialogues) ──
print('Loading ruslanmv/ai-medical-chatbot...')
ds1 = load_dataset('ruslanmv/ai-medical-chatbot', split='train').shuffle(seed=42).select(range(1500))
for r in ds1:
    q, a = str(r['Patient']).strip(), str(r['Doctor']).strip()
    if len(q) > 20 and len(a) > 40:
        english_qa.append((q, a))
print(f'  Loaded {len(english_qa)} rows')

# ── Dataset 2: lavita/medical-qa-datasets — health_advice subset ──
print('Loading lavita/medical-qa-datasets (health_advice)...')
try:
    ds2 = load_dataset('lavita/medical-qa-datasets', 'medical_meadow_health_advice', split='train').shuffle(seed=42).select(range(500))
    before = len(english_qa)
    for r in ds2:
        q, a = str(r.get('input', r.get('question', ''))).strip(), str(r.get('output', r.get('answer', ''))).strip()
        if len(q) > 20 and len(a) > 40:
            english_qa.append((q, a))
    print(f'  Loaded {len(english_qa) - before} rows')
except Exception as e:
    print(f'  Skipped: {e}')

# ── Dataset 3: lavita — mediqa (medical question answering) ──
print('Loading lavita/medical-qa-datasets (mediqa)...')
try:
    ds3 = load_dataset('lavita/medical-qa-datasets', 'medical_meadow_mediqa', split='train').shuffle(seed=42).select(range(300))
    before = len(english_qa)
    for r in ds3:
        q, a = str(r.get('input', r.get('question', ''))).strip(), str(r.get('output', r.get('answer', ''))).strip()
        if len(q) > 20 and len(a) > 40:
            english_qa.append((q, a))
    print(f'  Loaded {len(english_qa) - before} rows')
except Exception as e:
    print(f'  Skipped: {e}')

# Deduplicate and cap length for translation speed
MAX_CHARS = 400  # keep short for translation + training efficiency
english_qa = [(q, a) for q, a in english_qa if len(q) + len(a) <= MAX_CHARS]
english_qa = list({q: (q, a) for q, a in english_qa}.values())  # deduplicate by question
random.shuffle(english_qa)

print(f'\nTotal clean English pairs: {len(english_qa)}')

In [ ]:
# ============================================================
# CELL 4: Translate English → 4 Indian Languages
#
# We take 200 high-quality English Q&A pairs and translate
# them to each Indian language using IndicTrans2.
# 200 pairs × 4 languages = 800 Indian language samples.
# ============================================================
import gc

N_TRANSLATE = 200  # increase if you have more time/RAM
pairs_to_translate = english_qa[:N_TRANSLATE]
questions_en = [q for q, a in pairs_to_translate]
answers_en   = [a for q, a in pairs_to_translate]

translated_data = {}  # lang -> list of (q, a)

for lang, code in LANG_CODES.items():
    print(f'Translating {N_TRANSLATE} pairs to {lang} ({code})...')
    try:
        q_translated = translate_batch(questions_en, code, batch_size=8)
        a_translated = translate_batch(answers_en,   code, batch_size=8)
        translated_data[lang] = list(zip(q_translated, a_translated))
        print(f'  ✓ {lang}: {len(translated_data[lang])} pairs')
        print(f'  Sample Q: {q_translated[0]}')
        print(f'  Sample A: {a_translated[0][:100]}...')
    except Exception as e:
        print(f'  ✗ {lang} failed: {e}')
        translated_data[lang] = []

# Free translator from GPU memory before loading Sarvam
print('\nFreeing translator from GPU...')
del trans_model
del trans_tokenizer
gc.collect()
torch.cuda.empty_cache()
print('GPU memory freed!')

In [ ]:
# ============================================================
# CELL 5: Build Final Training Dataset
# ============================================================
from datasets import Dataset

PROMPT_TEMPLATE = '{instruction}\n\nPatient: {input}\nDoctor:'

INSTRUCTIONS = {
    'english': 'You are a helpful medical assistant. Answer the patient clearly and safely in English. Give detailed advice in 4-6 sentences. Always recommend consulting a doctor for serious issues.',
    'hindi':   'आप एक अनुभवी चिकित्सा सहायक हैं। मरीज के लक्षणों का उत्तर केवल हिंदी में दें। 4-6 वाक्यों में विस्तृत और उपयोगी सलाह दें।',
    'telugu':  'మీరు ఒక అనుభవజ్ఞుడైన వైద్య సహాయకుడు. రోగి లక్షణాలకు కేవలం తెలుగులో సమాధానం ఇవ్వండి. 4-6 వాక్యాలలో వివరంగా సలహా ఇవ్వండి.',
    'tamil':   'நீங்கள் ஒரு அனுபவமிக்க மருத்துவ உதவியாளர். நோயாளியின் அறிகுறிகளுக்கு தமிழிலேயே பதில் சொல்லுங்கள். 4-6 வாக்கியங்களில் விரிவான ஆலோசனை வழங்குங்கள்.',
    'kannada': 'ನೀವು ಅನುಭವಿ ವೈದ್ಯಕೀಯ ಸಹಾಯಕರು. ರೋಗಿಯ ಲಕ್ಷಣಗಳಿಗೆ ಕೇವಲ ಕನ್ನಡದಲ್ಲಿ ಉತ್ತರಿಸಿ. 4-6 ವಾಕ್ಯಗಳಲ್ಲಿ ವಿವರವಾಗಿ ಸಲಹೆ ನೀಡಿ.',
}

tokenizer_for_eos = None  # will be set after loading Sarvam tokenizer below

def make_rows_for_lang(lang, pairs):
    rows = []
    instruction = INSTRUCTIONS[lang]
    for q, a in pairs:
        q, a = str(q).strip(), str(a).strip()
        if len(q) < 10 or len(a) < 20:
            continue
        prompt = PROMPT_TEMPLATE.format(instruction=instruction, input=q)
        rows.append({'text': prompt + ' ' + a, 'lang': lang})
    return rows

all_rows = []

# English (all 1500+)
all_rows += make_rows_for_lang('english', english_qa)
print(f'English: {len(all_rows)}')

# Indian languages (translated)
for lang, pairs in translated_data.items():
    rows = make_rows_for_lang(lang, pairs)
    all_rows += rows
    print(f'{lang}: {len(rows)}')

random.shuffle(all_rows)
dataset = Dataset.from_list(all_rows)
print(f'\nTotal training samples: {len(dataset)}')

In [ ]:
# ============================================================
# CELL 6: Load Sarvam-2B + LoRA
# ============================================================
import os, torch
os.environ["BITSANDBYTES_NOWELCOME"] = "1"

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "sarvamai/sarvam-2b-v0.5"

# T4 = float16, A100/L4 = bfloat16
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

EOS = tokenizer.eos_token
dataset = dataset.map(lambda x: {"text": x["text"] + EOS, "lang": x["lang"]})

print("Loading Sarvam-2B (4-bit NF4)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",   # T4 does not support flash-attn
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print("✅ Sarvam-2B + LoRA ready!")


In [ ]:
# ============================================================
# CELL 7: Tokenize
# ============================================================
MAX_LENGTH = 512

def tokenize(examples):
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

tokenized = dataset.map(tokenize, batched=True, remove_columns=['text', 'lang'])
print(f'Tokenized: {len(tokenized)} samples')

In [ ]:
# ============================================================
# CELL 8: Fine-Tune
# ============================================================
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir='./sarvam_medical_v5',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=30,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_steps=300,
    save_total_limit=1,
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    report_to='none',
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

model.config.use_cache = False
print('Starting training...')
trainer.train()
trainer.save_model('./sarvam_medical_v5')
print('Training complete!')

In [ ]:
# ============================================================
# CELL 9: Inference + Test
# ============================================================
model.config.use_cache = True
model.eval()

STOP_WORDS = ['Patient:', 'Doctor:', '\n\nPatient', 'मरీज:', 'రోగి:', 'நோயாளி:', 'ರೋಗಿ:']

def detect_language(text):
    counts = {
        'hindi':   sum(1 for c in text if '\u0900' <= c <= '\u097F'),
        'telugu':  sum(1 for c in text if '\u0C00' <= c <= '\u0C7F'),
        'tamil':   sum(1 for c in text if '\u0B80' <= c <= '\u0BFF'),
        'kannada': sum(1 for c in text if '\u0C80' <= c <= '\u0CFF'),
    }
    best = max(counts, key=counts.get)
    return best if counts[best] > 0 else 'english'

def generate_response(user_msg, forced_lang=None):
    lang = forced_lang or detect_language(user_msg)
    prompt = PROMPT_TEMPLATE.format(instruction=INSTRUCTIONS[lang], input=user_msg)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    for stop in STOP_WORDS:
        if stop in reply:
            reply = reply[:reply.index(stop)].strip()
    return reply or '(No response)', lang

# Test all 5 languages
tests = [
    'I have had a fever and headache for two days. What should I do?',
    'मुझे दो दिनों से बुखार और सिरदर्द है। क्या करूं?',
    'నాకు రెండు రోజులుగా జ్వరం మరియు తలనొప్పి ఉంది. ఏమి చేయాలి?',
    'எனக்கு இரண்டு நாட்களாக காய்ச்சல் மற்றும் தலைவலி உள்ளது.',
    'ನನಗೆ ಎರಡು ದಿನಗಳಿಂದ ಜ್ವರ ಮತ್ತು ತಲೆನೋವು ಇದೆ.',
]
print('=' * 65)
for q in tests:
    ans, lang = generate_response(q)
    print(f'[{lang.upper()}]\nQ: {q}\nA: {ans}\n' + '-'*65)

In [ ]:
# ============================================================
# CELL 10: Gradio Chat UI
# ============================================================
import gradio as gr

LANG_MAP = {
    '🌐 Auto-detect': None,
    '🇺🇸 English':  'english',
    '🇮🇳 हिंदी':   'hindi',
    '🇮🇳 తెలుగు':  'telugu',
    '🇮🇳 தமிழ்':   'tamil',
    '🇮🇳 ಕನ್ನಡ':  'kannada',
}

with gr.Blocks(theme=gr.themes.Soft(), title='Medical Chatbot') as demo:
    gr.Markdown("""
# 🏥 Multilingual Medical Chatbot
Fine-tuned **Sarvam-2B** · Trained on real medical data translated via **IndicTrans2**

🇺🇸 English &nbsp;|&nbsp; 🇮🇳 हिंदी &nbsp;|&nbsp; 🇮🇳 తెలుగు &nbsp;|&nbsp; 🇮🇳 தமிழ் &nbsp;|&nbsp; 🇮🇳 ಕನ್ನಡ

> ⚠️ Educational only. Always consult a licensed doctor.
    """)
    with gr.Row():
        with gr.Column(scale=4):
            chatbot_ui = gr.Chatbot(label='Chat', height=500, type='messages', avatar_images=('👤','🏥'))
            with gr.Row():
                msg_box = gr.Textbox(placeholder='Describe your symptoms in any language...', show_label=False, scale=5, lines=2)
                send_btn = gr.Button('Send', variant='primary', scale=1)
            with gr.Row():
                detected_lbl = gr.Textbox(label='Detected Language', interactive=False, scale=2)
                clear_btn = gr.Button('Clear', variant='secondary', scale=1)
        with gr.Column(scale=1, min_width=200):
            lang_dd = gr.Dropdown(choices=list(LANG_MAP.keys()), value='🌐 Auto-detect', label='Force Language')
            gr.Markdown("""
### Try these:
**EN:** I have chest pain and breathlessness

**HI:** मेरे पेट में दर्द और उल्टी है

**TE:** నాకు కడుపు నొప్పి ఉంది

**TA:** வயிற்று வலி மற்றும் வாந்தி

**KA:** ನನಗೆ ತಲೆನೋವು ಮತ್ತು ವಾಂತಿ
            """)

    def respond(user_msg, history, lang_choice):
        if not user_msg.strip(): return history, '', ''
        forced = LANG_MAP.get(lang_choice)
        reply, detected = generate_response(user_msg, forced)
        history = list(history or [])
        history.append({'role': 'user', 'content': user_msg})
        history.append({'role': 'assistant', 'content': reply})
        return history, '', f'🌐 {detected.capitalize()}'

    msg_box.submit(respond, [msg_box, chatbot_ui, lang_dd], [chatbot_ui, msg_box, detected_lbl])
    send_btn.click(respond, [msg_box, chatbot_ui, lang_dd], [chatbot_ui, msg_box, detected_lbl])
    clear_btn.click(lambda: ([], '', ''), None, [chatbot_ui, msg_box, detected_lbl])

demo.launch(share=True, debug=False)